# 할거



- DB 재고 table

    | 상품명 | ID | 가격 | 재고 |
    | :---: | :---: | :---: | --- |
    | 라면 | 1 | 3,500 | 1 |
    | 김밥 | 2 | 3,500 | 2 |
    | 참치김밥 | 3 | 4,500 | 2 |
    | 돈까스 | 4 | 8,000 | 4 |
    | 치즈돈까스 | 5 | 8,500 | 3 |
    | 오므라이스 | 6 | 8,000 |2|

- 주문자 table

    | 주문번호 | 주문자 | 총 가격 | 전화번호 | 날짜 |
    | :---: | :---: | :---: | :---: | :---: |
    | 1001 | 김민수 | *** | 010-1231-1231 | 2026-07-29 12:23:23 |

- 주문 table

    | 주문번호 | 상품ID | 갯수 |
    | :---: | :---: | :---: |
    | 1001 | 1 | 1 |
    | 1001 | 2 | 1 |
    | 1001 | 4 | 1 |
    | 1002 | 1 | 1 |




- 개별 테이블 누르면 메뉴판에서 주문하는 식으로?
- 일단 첫 화면 인터페이스에서 테이블이랑 상품관리 같이
- 

# 

In [3]:
import pymysql

# DB 생성 및 연결

In [2]:
make_db_sql = '''
CREATE DATABASE IF NOT EXISTS inventorydb DEFAULT CHARACTER SET utf8mb4;

'''

In [3]:
conn = pymysql.connect(
    host="localhost",
    user="root",
    password="0000",
    charset="utf8"
)


In [4]:
cursor = conn.cursor()

In [5]:
cursor.execute(make_db_sql)

1

In [6]:
conn = conn = pymysql.connect(
    host="localhost",
    user="root",
    password="0000",
    database="inventorydb",
    charset="utf8"
)

In [7]:
cursor = conn.cursor()

In [8]:
cursor.execute("USE inventorydb")

0

In [9]:
cursor.execute("SELECT database()")

1

In [10]:
result = cursor.fetchone()

In [11]:
print("Database version:", result)

Database version: ('inventorydb',)


In [12]:
cursor.close()
conn.close()

# DB 테이블 생성

In [13]:
make_product_table_sql = '''
CREATE TABLE IF NOT EXISTS product (
  id INT PRIMARY KEY AUTO_INCREMENT,
  product_name VARCHAR(50) UNIQUE NOT NULL,
  price INT NOT NULL,
  stock INT NOT NULL
);
'''

In [14]:
make_customer_table_sql = '''
CREATE TABLE IF NOT EXISTS customer (
  customer_id INT PRIMARY KEY AUTO_INCREMENT,
  name VARCHAR(50) UNIQUE NOT NULL,
  total INT NOT NULL,
  phone VARCHAR(50) UNIQUE NOT NULL,
  datetime DATETIME NOT NULL
);
'''

In [18]:
make_order_table_sql = '''
CREATE TABLE IF NOT EXISTS orders (
  customer_id INT NOT NULL,
  FOREIGN KEY (customer_id) REFERENCES  customer(customer_id),
  product_id INT  NOT NULL,
  FOREIGN KEY (product_id) REFERENCES product(id),
  number INT NOT NULL
);
'''

In [2]:
DB_CONFIG = dict(
    host="localhost",
    user="root",
    password="0000",
    database="inventorydb",
    charset="utf8"
)

In [16]:
conn = pymysql.connect(
    host="localhost",
    user="root",
    password="0000",
    database="inventorydb",
    charset="utf8"
)
cursor = conn.cursor()


In [ ]:
cursor.execute(make_product_table_sql)
cursor.execute(make_customer_table_sql)
cursor.execute(make_order_table_sql)

0

In [20]:
cursor.close()
conn.close()

# DB 클래스 

- 상품 추가
- 재고 추가
- 재고 조회

- 재고 변경
- 주문 조회
- 

    소비자 테이블 생성
    제품 테이블에서 가능한지 확인:
        주문테이블 생성        
        제품 테이블에서 계산
        return 
    안된다:        
        return 

- 

In [6]:
class DB:
    def __init__(self, **config):
        self.config = config

    def connect(self):
        return pymysql.connect(**self.config)

    # 새 상품 추가
    def insert_product(self, product_name, price, stock):
        sql = "INSERT INTO product (product_name,price,stock) VALUES (%s,%s,%s)"
        with self.connect() as conn:
            try:
                with conn.cursor() as cursor:
                    cursor.execute(sql, (product_name, price, stock))
                conn.commit()
                return True
            except Exception as error:
                conn.rollback()
                print("상품 등록 실패:", repr(error))
                return False
    # 재고 추가
    def insert_stock(self, product_name,  number):
        sql = "UPDATE product SET stock = (%s) + stock WHERE product_name = (%s)"
        with self.connect() as conn:
            try:
                with conn.cursor() as cursor:
                    cursor.execute(sql, (number , product_name))
                conn.commit()
                return True
            except Exception:
                conn.rollback()
                return 
    # 모든 제품 조회
    def fetch_product(self):
            sql = "SELECT * FROM product"
            with self.connect() as conn:
                with conn.cursor() as cursor:
                    cursor.execute(sql)
                    return cursor.fetchall()  # [(customer_id, product_name, number, datetime)]

    # 재고 조회
    def fetch_stock(self, product_name):
        sql = "SELECT stock FROM product = %s"
        with self.connect() as conn:
            with conn.cursor() as cursor:
                cursor.execute(sql,product_name)
                return cursor.fetchone()  # [(stock),]
    # 재고 사용
    def update_stock(self, product_name,  number):
        sql = "UPDATE product SET stock = stock -(%s)  WHERE product_name = (%s)"
        with self.connect() as conn:
            try:
                with conn.cursor() as cursor:
                    cursor.execute(sql, (number , product_name))
                conn.commit()
                return True
            except Exception:
                conn.rollback()
                return     
    # 상세 주문 추가 - 주문 테이블 추가
    def insert_order(self, customer_id, product_name, number):
        sql = "INSERT INTO orders (customer_id, product_name,number) VALUES (%s,%s,%s)"
        with self.connect() as conn:
            try:
                with conn.cursor() as cursor:
                    cursor.execute(sql, (customer_id, product_name, number))
                conn.commit()
                return True
            except Exception:
                conn.rollback()
                return False

    # 주문 정보
    def insert_customer(self, customer_id, name, total, phone, datetime):
            sql = "INSERT INTO customer (customer_id, name,total,phone,datetime) VALUES (%s,%s,%s,%s,%s)"
            with self.connect() as conn:
                try:
                    with conn.cursor() as cursor:
                        cursor.execute(sql, (customer_id, name, total, phone,datetime))
                    conn.commit()
                    return True
                except Exception:
                    conn.rollback()
                    return False

    # 주문 조회
    # def fetch_order(self, product_name, number):
    #     sql = "SELECT * FROM customers (product_name,number) VALUES (%s,%s)"
    #     with self.connect() as conn:
    #         with conn.cursor() as cursor:
    #             cursor.execute(sql,product_name, number)
    #             return cursor.fetchall()  # [(customer_id, product_name, number, datetime)]

# QT

## 주문

In [4]:
from PyQt5.QtWidgets import QMainWindow, QVBoxLayout, QFormLayout, QLineEdit, QPushButton, QMessageBox, QSpinBox
# from db_helper import DB, DB_CONFIG


In [5]:
from datetime import date

In [6]:
str(date.today())

'2026-07-30'

In [8]:
class OrderDialog(QMainWindow):
    def __init__(self,parent=None):
        super().__init__(parent)
        self.setWindowTitle("주문")
        self.db = DB(**DB_CONFIG)

        self.number = QLineEdit()
        self.username = QLineEdit()
        self.phonenumber = QLineEdit()

        products = DB.fetch_product
        form = QFormLayout()
        for product_name in products:
            form.addRow(product_name[1],self.number)
        form.addRow("이름", self.username)
        form.addRow("전화번호", self.phonenumber)

        self.btn_order = QPushButton("주문")
        self.btn_order.clicked.connect(self.try_order)

        layout = QVBoxLayout()
        layout.addLayout(form)
        layout.addWidget(self.btn_login)
        self.setLayout(layout)            

    def try_order(self):
        total = 0
        name = self.username.text().strip()
        phone = self.phonenumber.text().strip()
        for product in products:
            number = int(self.number.text())
            if self.db.fetch_stock(product[1]) < number:
                QMessageBox.warning(self, product[1]+"의 재고가 부족합니다", product[3]+"개 이하로 다시 시도해주세요")
                return
            else: 
                # 계산하고, 재고 빼고, 주문서 테이블 추가 
                self.db.update_stock(product[1],number)
                total += product[2] * number
                self.db.insert_customer(name,total,phone,str(date.today()))
                self.db.insert_order(product[1],number)
                QMessageBox(self, "주문이 완료되었습니다")

C:\Users\302-16\AppData\Local\Temp\ipykernel_6960\1562996102.py:1: DeprecationWarning: sipPyTypeDict() is deprecated, the extension module should use sipPyTypeDictRef() instead
  class OrderDialog(QMainWindow):


In [ ]:
# main_window.py
from PyQt5.QtWidgets import QMainWindow, QWidget, QVBoxLayout, QHBoxLayout, QTableWidget, QTableWidgetItem, \
    QLabel, QLineEdit, QPushButton, QMessageBox
from db_helper import DB, DB_CONFIG

class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("회원 관리")
        self.db = DB(**DB_CONFIG)

        # 중앙 위젯 및 레이아웃
        central = QWidget()
        self.setCentralWidget(central)
        vbox = QVBoxLayout(central)

        # 상단: 입력 폼 + 추가 버튼
        form_box = QHBoxLayout()
        self.input_name = QLineEdit()
        self.input_email = QLineEdit()
        self.btn_add = QPushButton("추가")
        self.btn_add.clicked.connect(self.add_member)

        form_box.addWidget(QLabel("이름"))
        form_box.addWidget(self.input_name)
        form_box.addWidget(QLabel("이메일"))
        form_box.addWidget(self.input_email)
        form_box.addWidget(self.btn_add)

        # 중앙: 테이블 위젯
        self.table = QTableWidget()
        self.table.setColumnCount(3)
        self.table.setHorizontalHeaderLabels(["ID", "이름", "이메일"])
        self.table.setEditTriggers(self.table.NoEditTriggers)  # 표준 예시: 목록은 읽기 전용
        self.table.verticalHeader().setVisible(False)

        # 배치
        vbox.addLayout(form_box)
        vbox.addWidget(self.table)

        # 초기 데이터 로드
        self.load_members()

    def load_members(self):
        rows = self.db.fetch_members()
        self.table.setRowCount(len(rows))
        for r, (mid, name, email) in enumerate(rows):
            self.table.setItem(r, 0, QTableWidgetItem(str(mid)))
            self.table.setItem(r, 1, QTableWidgetItem(name))
            self.table.setItem(r, 2, QTableWidgetItem(email))
        self.table.resizeColumnsToContents()

    def add_member(self):
        name = self.input_name.text().strip()
        email = self.input_email.text().strip()
        if not name or not email:
            QMessageBox.warning(self, "오류", "이름과 이메일을 모두 입력하세요.")
            return
        ok = self.db.insert_member(name, email)
        if ok:
            QMessageBox.information(self, "완료", "추가되었습니다.")
            self.input_name.clear()
            self.input_email.clear()
            self.load_members()
        else:
            QMessageBox.critical(self, "실패", "추가 중 오류가 발생했습니다.")

In [1]:
import sys
from PyQt5.QtWidgets import QApplication

In [12]:
import sys
from PyQt5.QtWidgets import QApplication

if __name__ == "__main__":
    app = QApplication(sys.argv)
    w = OrderDialog()
    w.show()
    sys.exit(app.exec_())


: 

In [8]:
app = QApplication(sys.argv)
w = OrderDialog()
w.show()
sys.exit(app.exec_())

: 

In [7]:
db = DB(**DB_CONFIG)

In [8]:
sql = "INSERT INTO product (product_name,price,stock) VALUES (%s,%s,%s)"

In [73]:
db.connect()

In [9]:
conn = db.connect()

In [10]:
cursor = conn.cursor()

In [57]:
conn.rollback()
cursor.close()
conn.close()

InterfaceError: (0, '')

In [63]:
with db.connect() as conn:
    try:
        with conn.cursor() as cursor:
            cursor.execute(sql, ("돈까스",8000,5))
        conn.commit()

    except Exception:
        conn.rollback()


In [16]:
db.insert_product("참치김밥",4500,30)

True

In [81]:
ok

True

In [17]:
products = db.fetch_product()

In [18]:
len(products)

6

In [19]:
products

((1, '떡볶이', 2000, 4),
 (2, '라면', 3500, 10),
 (3, '돈까스', 8000, 15),
 (4, '치즈돈까스', 8500, 15),
 (5, '김밥', 3500, 30),
 (6, '참치김밥', 4500, 30))

In [84]:
for product in products:
    print(product[1])

라면
치킨
돈까스
햄버거
떡볶이
